# Notebook 1: Data Loading and Preprocessing

This notebook:
- Loads the master panel dataset (POLICYUS.xlsx — 141 variables, 1987–2023)
- Checks for null values
- Applies stationarity transformations (log, Box-Cox, Yeo-Johnson)
- Saves the cleaned dataset as df_transformed.csv

In [26]:
import pandas as pd
import numpy as np
from scipy.stats import boxcox, yeojohnson
from statsmodels.tsa.stattools import adfuller
import warnings
warnings.filterwarnings('ignore')
print('Libraries loaded.')

Libraries loaded.


In [27]:
# Load master dataset
df = pd.read_excel('../data/raw/POLICYUS.xlsx', parse_dates=['Date'], index_col='Date')
print(f'Shape: {df.shape}')
print(f'Date range: {df.index[0].date()} to {df.index[-1].date()}')

Shape: (435, 140)
Date range: 1987-04-01 to 2023-06-01


In [28]:
# Check nulls
print(f'Total missing values: {df.isnull().sum().sum()}')
df[['EPU', 'CPU', 'GPR']].describe().round(2)

Total missing values: 0


,EPU,CPU,GPR
count,435.00,435.00,435.00
mean,124.59,102.80,100.15
std,59.00,58.11,48.88
min,44.78,28.16,39.05
25%,85.33,64.89,78.04
50%,107.95,87.95,89.74
75%,152.63,112.47,108.39
max,503.96,411.29,512.53


In [29]:
# Variables that need transformation
non_stationary_cols = [
    'CPU', 'CES0600000007', 'HOUST', 'HOUSTNE', 'HOUSTMW', 'HOUSTS', 'HOUSTW',
    'PERMIT', 'PERMITNE', 'PERMITMW', 'PERMITS', 'PERMITW', 'CAPE', 'cred',
    'credgdp', 'capr', 'mortg_inc', 'mortg', 'prfi_gdp', 'pip_inc'
]

def best_transform(df, columns):
    df_t = df.copy()
    log = {}
    for col in columns:
        s = df[col].dropna()
        results = {}
        if (s > 0).all():
            results['log'] = adfuller(np.log(s))[1]
            bc, _ = boxcox(s)
            results['boxcox'] = adfuller(bc)[1]
        yj, _ = yeojohnson(s)
        results['yeojohnson'] = adfuller(yj)[1]
        best = min(results, key=results.get)
        if results[best] < 0.05:
            if best == 'log':
                df_t[col] = np.log(df[col])
            elif best == 'boxcox':
                transformed, _ = boxcox(s)
                df_t.loc[s.index, col] = transformed
            elif best == 'yeojohnson':
                transformed, _ = yeojohnson(s)
                df_t.loc[s.index, col] = transformed
            log[col] = f'{best} (p={results[best]:.4f})'
        else:
            df_t[col] = df[col].diff()
            log[col] = 'differencing'
    return df_t, log

df_transformed, transform_log = best_transform(df, non_stationary_cols)
df_transformed = df_transformed.dropna()
print('Transformations applied:')
for col, method in transform_log.items():
    print(f'  {col}: {method}')

Transformations applied:
  CPU: differencing
  CES0600000007: differencing
  HOUST: differencing
  HOUSTNE: differencing
  HOUSTMW: differencing
  HOUSTS: differencing
  HOUSTW: differencing
  PERMIT: differencing
  PERMITNE: differencing
  PERMITMW: differencing
  PERMITS: differencing
  PERMITW: differencing
  CAPE: differencing
  cred: yeojohnson (p=0.0223)
  credgdp: differencing
  capr: differencing
  mortg_inc: differencing
  mortg: yeojohnson (p=0.0442)
  prfi_gdp: differencing
  pip_inc: differencing


In [30]:
# Save transformed dataset
df_transformed.to_csv('../data/processed/df_transformed.csv')
print(f'Saved. Shape: {df_transformed.shape}')

Saved. Shape: (434, 140)
